# NHS Referral-to-Treatment Waiting Times Analysis
## Multi-Month Data Preparation

**Analysis window:** April 2025 to July 2026  
**Dataset:** NHS England Consultant-led Referral to Treatment (RTT) — Incomplete Provider data

### Purpose

`01_data_understanding.ipynb` established the raw structure and `02_data_cleaning.ipynb` cleaned July 2026. This notebook generalises that cleaning process across **16 monthly extracts** so later analysis can answer trend questions rather than describe a single month.

It will:

- discover monthly raw workbooks;
- infer each reporting month from the filename;
- detect the real header row automatically;
- apply the same cleaning rules to every month;
- validate each file independently;
- check for missing reporting periods;
- combine the monthly files; and
- export trend-ready CSV datasets for SQL, Python and Power BI.

> Raw NHS files are never overwritten.

In [10]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

## Project folders and reporting window

Place monthly workbooks in `../data/raw/monthly/`. The notebook also checks `../data/raw/`, so the July 2026 workbook already used in the project can remain where it is.

In [11]:
raw_root = Path("../data/raw")
monthly_dir = raw_root / "monthly"
processed_dir = Path("../data/processed")

monthly_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

start_month = pd.Timestamp("2025-04-01")
end_month = pd.Timestamp("2026-07-01")
expected_months = pd.date_range(start_month, end_month, freq="MS")

print("Expected monthly files:", len(expected_months))
print("First month:", expected_months[0].strftime("%B %Y"))
print("Last month:", expected_months[-1].strftime("%B %Y"))

Expected monthly files: 16
First month: April 2025
Last month: July 2026


We use a continuous **16-month period**. April 2025–March 2026 comes from the NHS England 2025/26 RTT archive, while April–July 2026 comes from the 2026/27 archive. Where NHS England marks a file as revised, use that revised file.

## Infer the reporting month from each filename

The parser looks for codes such as `Apr25`, `Dec25`, `Mar26`, or `Jul26`, so extra filename text does not matter.

In [12]:
month_lookup = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4,
    "may": 5, "jun": 6, "jul": 7, "aug": 8,
    "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

def infer_reporting_month(filename):
    match = re.search(
        r"(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[-_ ]?(\d{2})",
        filename,
        flags=re.IGNORECASE
    )
    if not match:
        return pd.NaT

    month = month_lookup[match.group(1).lower()]
    year = 2000 + int(match.group(2))
    return pd.Timestamp(year=year, month=month, day=1)

## Detect the actual header row

Instead of assuming every workbook has its header on Excel row 14, the function searches the top of the `Provider` sheet for `Provider Code`. This protects the workflow from small formatting changes.

In [13]:
def detect_header_row(file_path, sheet_name="Provider", max_rows=30):
    preview = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=None,
        nrows=max_rows
    )

    for row_index in range(len(preview)):
        values = preview.iloc[row_index].astype("string").str.strip()
        if values.eq("Provider Code").any():
            return row_index

    raise ValueError(
        f"Could not find the Provider Code header in {file_path.name}"
    )

## Percentage helper

The July workbook stored percentages as proportions such as `0.7205`. This helper safely supports both decimal proportions and already-scaled percentage values.

In [14]:
def clean_percentage(series):
    numeric = pd.to_numeric(series, errors="coerce")
    non_missing = numeric.dropna()

    if non_missing.empty:
        return numeric

    if non_missing.abs().max() <= 1.5:
        numeric = numeric * 100

    return numeric.round(2)

## Reusable cleaning and validation function

This function applies the decisions from `02_data_cleaning.ipynb` to each monthly workbook and validates the result before it is allowed into the combined trend dataset.

In [15]:
def clean_monthly_provider_file(file_path):
    reporting_month = infer_reporting_month(file_path.name)
    if pd.isna(reporting_month):
        raise ValueError(f"Could not infer reporting month from {file_path.name}")

    header_row = detect_header_row(file_path)
    df = pd.read_excel(file_path, sheet_name="Provider", header=header_row)
    df = df.dropna(axis=1, how="all")

    weekly_raw = [
        col for col in df.columns
        if re.match(r"^>\d+-\d+$", str(col)) or col == "104 plus"
    ]

    if len(weekly_raw) != 105:
        raise ValueError(
            f"{file_path.name}: expected 105 weekly bands, found {len(weekly_raw)}"
        )

    rename_map = {
        "Region Code": "region_code",
        "Provider Code": "provider_code",
        "Provider Name": "provider_name",
        "Treatment Function Code": "treatment_function_code",
        "Treatment Function": "treatment_function",
        "Total number of incomplete pathways": "total_waiting",
        "Total within 18 weeks": "within_18_weeks",
        "% within 18 weeks": "pct_within_18_weeks",
        "Average (median) waiting time (in weeks)": "median_wait_weeks",
        "92nd percentile waiting time (in weeks)": "p92_wait_weeks",
        "Total 52 plus weeks": "over_52_weeks",
        "Total 65 plus weeks": "over_65_weeks",
        "Total 78 plus weeks": "over_78_weeks",
        "% 52 plus weeks": "pct_over_52_weeks",
    }

    for col in weekly_raw:
        if col == "104 plus":
            rename_map[col] = "wait_104_plus_weeks"
        else:
            start, end = re.findall(r"\d+", col)
            rename_map[col] = f"wait_{start}_{end}_weeks"

    df = df.rename(columns=rename_map)

    required = [
        "provider_code", "provider_name",
        "treatment_function_code", "treatment_function",
        "total_waiting", "within_18_weeks",
        "pct_within_18_weeks", "median_wait_weeks",
        "p92_wait_weeks", "over_52_weeks",
        "over_65_weeks", "over_78_weeks",
        "pct_over_52_weeks"
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{file_path.name}: missing columns {missing}")

    for col in ["median_wait_weeks", "p92_wait_weeks"]:
        values = df[col].astype("string").str.strip()
        df[col] = pd.to_numeric(
            values.mask(values.eq("-")),
            errors="coerce"
        )

    df["pct_within_18_weeks"] = clean_percentage(df["pct_within_18_weeks"])
    df["pct_over_52_weeks"] = clean_percentage(df["pct_over_52_weeks"])

    df["over_18_weeks"] = df["total_waiting"] - df["within_18_weeks"]
    df["pct_over_18_weeks"] = (100 - df["pct_within_18_weeks"]).round(2)

    df["reporting_month"] = reporting_month
    df["reporting_period"] = reporting_month.strftime("%Y-%m")
    df["source_file"] = file_path.name
    df["row_type"] = np.where(
        df["treatment_function_code"].eq("C_999"),
        "provider_total",
        "treatment_function"
    )

    weekly_clean = [rename_map[col] for col in weekly_raw]

    id_columns = [
        "reporting_month", "reporting_period", "source_file",
        "region_code", "provider_code", "provider_name",
        "treatment_function_code", "treatment_function", "row_type"
    ]

    kpi_columns = [
        "total_waiting", "within_18_weeks", "over_18_weeks",
        "pct_within_18_weeks", "pct_over_18_weeks",
        "median_wait_weeks", "p92_wait_weeks",
        "over_52_weeks", "over_65_weeks", "over_78_weeks",
        "pct_over_52_weeks"
    ]

    df = df[id_columns + weekly_clean + kpi_columns]

    # validation
    if df.duplicated(["provider_code", "treatment_function_code"]).sum() != 0:
        raise ValueError(f"{file_path.name}: duplicate provider/treatment-function rows")

    if not df[weekly_clean].sum(axis=1).eq(df["total_waiting"]).all():
        raise ValueError(f"{file_path.name}: weekly bands do not match total_waiting")

    within_18_cols = [f"wait_{w}_{w+1}_weeks" for w in range(18)]
    if not df[within_18_cols].sum(axis=1).eq(df["within_18_weeks"]).all():
        raise ValueError(f"{file_path.name}: 18-week bands do not reconcile")

    long_wait_checks = {
        "over_52_weeks": weekly_clean[52:],
        "over_65_weeks": weekly_clean[65:],
        "over_78_weeks": weekly_clean[78:],
    }
    for published_col, band_cols in long_wait_checks.items():
        if not df[band_cols].sum(axis=1).eq(df[published_col]).all():
            raise ValueError(f"{file_path.name}: {published_col} does not reconcile")

    count_columns = weekly_clean + [
        "total_waiting", "within_18_weeks", "over_18_weeks",
        "over_52_weeks", "over_65_weeks", "over_78_weeks"
    ]
    if (df[count_columns] < 0).any().any():
        raise ValueError(f"{file_path.name}: negative pathway counts detected")

    return df

## Discover the monthly raw files

In [16]:
candidate_files = sorted(
    set(monthly_dir.glob("*.xlsx")) | set(raw_root.glob("*.xlsx"))
)

file_inventory = []
for file_path in candidate_files:
    reporting_month = infer_reporting_month(file_path.name)
    if pd.notna(reporting_month):
        file_inventory.append({
            "reporting_month": reporting_month,
            "reporting_period": reporting_month.strftime("%Y-%m"),
            "file_name": file_path.name,
            "path": file_path,
        })

inventory_df = pd.DataFrame(file_inventory)
if not inventory_df.empty:
    inventory_df = inventory_df.sort_values("reporting_month").reset_index(drop=True)

inventory_df

,reporting_month,reporting_period,file_name,path
0,2025-04-01,2025-04,Incomplete-Provider-Apr25-XLSX-9M-revised.xlsx,../data/raw/monthly/Incomplete-Provider-Apr25-XLSX-9M-revised.xlsx
1,2025-05-01,2025-05,Incomplete-Provider-May25-XLSX-9M-revised.xlsx,../data/raw/monthly/Incomplete-Provider-May25-XLSX-9M-revised.xlsx
2,2025-06-01,2025-06,Incomplete-Provider-Jun25-XLSX-9M-revised.xlsx,../data/raw/monthly/Incomplete-Provider-Jun25-XLSX-9M-revised.xlsx
3,2025-07-01,2025-07,Incomplete-Provider-Jul25-XLSX-9M-revised-2.xlsx,../data/raw/monthly/Incomplete-Provider-Jul25-XLSX-9M-revised-2.xlsx
4,2025-08-01,2025-08,Incomplete-Provider-Aug25-XLSX-9M-revised-2.xlsx,../data/raw/monthly/Incomplete-Provider-Aug25-XLSX-9M-revised-2.xlsx
5,2025-09-01,2025-09,Incomplete-Provider-Sep25-XLSX-9M-revised.xlsx,../data/raw/monthly/Incomplete-Provider-Sep25-XLSX-9M-revised.xlsx
6,2025-10-01,2025-10,Incomplete-Provider-Oct25-XLSX-9M-SrRW6y.xlsx,../data/raw/monthly/Incomplete-Provider-Oct25-XLSX-9M-SrRW6y.xlsx
7,2025-11-01,2025-11,Incomplete-Provider-Nov25-XLSX-9M-1Xmjkk.xlsx,../data/raw/monthly/Incomplete-Provider-Nov25-XLSX-9M-1Xmjkk.xlsx
8,2025-12-01,2025-12,Incomplete-Provider-Dec25-XLSX-9M-6jPlxd.xlsx,../data/raw/monthly/Incomplete-Provider-Dec25-XLSX-9M-6jPlxd.xlsx
9,2026-01-01,2026-01,Incomplete-Provider-Jan26-XLSX-9M-WL5BiP.xlsx,../data/raw/monthly/Incomplete-Provider-Jan26-XLSX-9M-WL5BiP.xlsx


## Check for missing months

A trend dataset with silent gaps can produce misleading conclusions, so this notebook explicitly checks the expected 16 reporting periods before combining anything.

In [17]:
available_months = set(inventory_df["reporting_month"]) if not inventory_df.empty else set()
missing_months = [m for m in expected_months if m not in available_months]

print("Expected months:", len(expected_months))
print("Available months:", len(available_months))
print("Missing months:", len(missing_months))

if missing_months:
    print("\nStill needed:")
    for month in missing_months:
        print("-", month.strftime("%B %Y"))
else:
    print("\nAll required monthly files are available.")

Expected months: 16
Available months: 16
Missing months: 0

All required monthly files are available.


### Download checklist

For every missing month, download the official **Incomplete Provider** workbook from NHS England and place it in `data/raw/monthly/`. Use revised versions where NHS England marks them as revised.

The filename can retain its original NHS wording; it only needs to contain a code such as `Apr25` or `Jul26` so the reporting month can be inferred.

## Clean and combine all 16 months

This cell deliberately stops if any required reporting month is missing.

In [18]:
if missing_months:
    missing_labels = ", ".join(m.strftime("%b %Y") for m in missing_months)
    raise RuntimeError(
        "The 16-month dataset is incomplete. "
        f"Download these months first: {missing_labels}"
    )

cleaned_months = []

for _, row in inventory_df.iterrows():
    month = row["reporting_month"]
    if start_month <= month <= end_month:
        print(f"Cleaning {row['file_name']}...")
        cleaned_months.append(clean_monthly_provider_file(row["path"]))

combined_df = pd.concat(cleaned_months, ignore_index=True)
combined_df.shape

Cleaning Incomplete-Provider-Apr25-XLSX-9M-revised.xlsx...


/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["over_18_weeks"] = df["total_waiting"] - df["within_18_weeks"]
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["pct_over_18_weeks"] = (100 - df["pct_within_18_weeks"]).round(2)
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usu

Cleaning Incomplete-Provider-May25-XLSX-9M-revised.xlsx...


/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["over_18_weeks"] = df["total_waiting"] - df["within_18_weeks"]
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["pct_over_18_weeks"] = (100 - df["pct_within_18_weeks"]).round(2)
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usu

Cleaning Incomplete-Provider-Jun25-XLSX-9M-revised.xlsx...


/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["over_18_weeks"] = df["total_waiting"] - df["within_18_weeks"]
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["pct_over_18_weeks"] = (100 - df["pct_within_18_weeks"]).round(2)
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usu

... (23 outputs left out)

/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["over_18_weeks"] = df["total_waiting"] - df["within_18_weeks"]
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["pct_over_18_weeks"] = (100 - df["pct_within_18_weeks"]).round(2)
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usu

Cleaning Incomplete-Provider-Jul26-XLSX-9M-97ylx5.xlsx...


/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["over_18_weeks"] = df["total_waiting"] - df["within_18_weeks"]
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["pct_over_18_weeks"] = (100 - df["pct_within_18_weeks"]).round(2)
/var/folders/zd/0bzshzyx5tx0n2wd8q88c7480000gn/T/ipykernel_7786/210276591.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usu

(57480, 125)

## Validate the combined dataset

Provider coverage can legitimately change between months, so we do not assume every period has exactly the same number of trusts. Instead, uniqueness is tested on:

**reporting month + provider code + treatment-function code**

In [19]:
combined_duplicates = combined_df.duplicated(
    ["reporting_month", "provider_code", "treatment_function_code"]
).sum()

print("Duplicate month/provider/treatment-function rows:", combined_duplicates)
assert combined_duplicates == 0

actual_months = pd.DatetimeIndex(combined_df["reporting_month"].unique()).sort_values()
assert len(actual_months) == len(expected_months)
assert actual_months[0] == start_month
assert actual_months[-1] == end_month

print("Combined reporting-period coverage is complete.")

Duplicate month/provider/treatment-function rows: 0
Combined reporting-period coverage is complete.


## Monthly structural coverage

In [20]:
monthly_coverage = (
    combined_df
    .groupby(["reporting_month", "reporting_period"], as_index=False)
    .agg(
        rows=("provider_code", "size"),
        providers=("provider_code", "nunique"),
        treatment_functions=("treatment_function_code", "nunique"),
        provider_total_rows=(
            "row_type",
            lambda s: (s == "provider_total").sum()
        )
    )
)

monthly_coverage

,reporting_month,reporting_period,rows,providers,treatment_functions,provider_total_rows
0,2025-04-01,2025-04,3648,152,24,152
1,2025-05-01,2025-05,3648,152,24,152
2,2025-06-01,2025-06,3648,152,24,152
3,2025-07-01,2025-07,3600,150,24,150
4,2025-08-01,2025-08,3600,150,24,150
5,2025-09-01,2025-09,3600,150,24,150
6,2025-10-01,2025-10,3600,150,24,150
7,2025-11-01,2025-11,3576,149,24,149
8,2025-12-01,2025-12,3600,150,24,150
9,2026-01-01,2026-01,3600,150,24,150


## Create a provider-total monthly table

For provider-level trends, use only the `C_999` rows. This avoids double-counting the treatment-function rows.

In [21]:
provider_monthly = combined_df[
    combined_df["row_type"] == "provider_total"
].copy()

provider_monthly[[
    "reporting_period",
    "provider_code",
    "provider_name",
    "total_waiting",
    "pct_within_18_weeks",
    "median_wait_weeks",
    "p92_wait_weeks",
    "over_52_weeks",
    "over_65_weeks",
    "over_78_weeks"
]].head()

,reporting_period,provider_code,provider_name,total_waiting,pct_within_18_weeks,median_wait_weeks,p92_wait_weeks,over_52_weeks,over_65_weeks,over_78_weeks
23,2025-04,R1H,BARTS HEALTH NHS TRUST,123189,55.42,14.874405,44.1213,3675,333,29
47,2025-04,R1K,LONDON NORTH WEST UNIVERSITY HEALTHCARE NHS TRUST,85677,52.21,16.540356,45.037165,2863,25,0
71,2025-04,RAL,ROYAL FREE LONDON NHS FOUNDATION TRUST,149100,56.32,14.635166,42.773485,3896,105,4
95,2025-04,RAN,ROYAL NATIONAL ORTHOPAEDIC HOSPITAL NHS TRUST,8454,69.12,10.75976,33.516164,48,6,0
119,2025-04,RAS,THE HILLINGDON HOSPITALS NHS FOUNDATION TRUST,31043,53.73,15.68134,40.625683,431,2,0


## Gloucestershire Hospitals continuity check

In [22]:
gloucestershire_trend = (
    provider_monthly[provider_monthly["provider_code"] == "RTE"]
    [[
        "reporting_month",
        "reporting_period",
        "provider_name",
        "total_waiting",
        "within_18_weeks",
        "over_18_weeks",
        "pct_within_18_weeks",
        "median_wait_weeks",
        "p92_wait_weeks",
        "over_52_weeks",
        "over_65_weeks",
        "over_78_weeks"
    ]]
    .sort_values("reporting_month")
)

gloucestershire_trend

,reporting_month,reporting_period,provider_name,total_waiting,within_18_weeks,over_18_weeks,pct_within_18_weeks,median_wait_weeks,p92_wait_weeks,over_52_weeks,over_65_weeks,over_78_weeks
839,2025-04-01,2025-04,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,66826,46113,20713,69.00,9.901021,34.298955,93,3,1
4487,2025-05-01,2025-05,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,62970,43202,19768,68.61,10.284866,34.805572,75,3,0
8135,2025-06-01,2025-06,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,60281,41530,18751,68.89,10.030578,34.581461,46,2,1
11783,2025-07-01,2025-07,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,59146,40167,18979,67.91,10.494444,35.06094,39,3,1
15383,2025-08-01,2025-08,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,59797,40215,19582,67.25,10.737718,34.750652,36,2,2
18983,2025-09-01,2025-09,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,63422,44241,19181,69.76,10.430212,33.79356,35,1,0
22583,2025-10-01,2025-10,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,62288,43213,19075,69.38,9.874251,34.22413,28,0,0
26159,2025-11-01,2025-11,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,63147,44200,18947,70.00,9.955429,34.000416,31,0,0
29759,2025-12-01,2025-12,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,62157,43056,19101,69.27,10.160151,34.347922,30,0,0
33359,2026-01-01,2026-01,GLOUCESTERSHIRE HOSPITALS NHS FOUNDATION TRUST,62862,43065,19797,68.51,10.375927,34.323264,21,1,0


## Export trend-ready datasets

In [23]:
combined_output = processed_dir / "rtt_incomplete_provider_apr2025_jul2026_clean.csv"
provider_output = processed_dir / "rtt_provider_totals_apr2025_jul2026.csv"

combined_df.to_csv(combined_output, index=False)
provider_monthly.to_csv(provider_output, index=False)

print("Saved full dataset:", combined_output)
print("Saved provider totals:", provider_output)
print(f"Full rows: {len(combined_df):,}")
print(f"Provider-total rows: {len(provider_monthly):,}")

Saved full dataset: ../data/processed/rtt_incomplete_provider_apr2025_jul2026_clean.csv
Saved provider totals: ../data/processed/rtt_provider_totals_apr2025_jul2026.csv
Full rows: 57,480
Provider-total rows: 2,395
